# 05 · Key Insights & Recommendations
สรุปตัวเลขหลักจาก notebook 01–04 และข้อเสนอทางการตลาด
(รัน notebook 01–04 ก่อน เพื่อสร้างไฟล์ใน `data/processed/`)

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.viz import setup, save, seg_palette, PRIMARY, ACCENT, MUTED
setup()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.width", 160)

In [2]:
from src.data import load, load_transactions_clean

tx = load_transactions_clean()
seg = pd.read_parquet(config.PROCESSED_DIR / "segments.parquet")
fwd = pd.read_parquet(config.PROCESSED_DIR / "forward_validation.parquet")
migr = pd.read_parquet(config.PROCESSED_DIR / "migration.parquet")
lift_all = pd.read_parquet(config.PROCESSED_DIR / "campaign_lift_overall.parquet")
lift_type = pd.read_parquet(config.PROCESSED_DIR / "campaign_lift_by_type.parquet")
ct, redempt = load("campaign_table"), load("coupon_redempt")
ORDER, TH = config.SEGMENT_ORDER, config.SEGMENT_TH
tx["segment"] = tx.household_key.map(seg.segment)

In [3]:
weekly = tx.groupby("WEEK_NO").SALES_VALUE.sum()
merch = tx[~tx.is_non_merch]

s = seg.groupby("segment")
seg_tbl = pd.DataFrame({
    "ชื่อกลุ่ม": pd.Series(TH),
    "households": s.size(),
    "% sales": s.monetary.sum() / seg.monetary.sum() * 100,
    "median visits / wk": s.baskets_per_week.median(),
    "median CLV 12m $": s.clv_12m.median(),
    "% of campaign sends": ct.household_key.map(seg.segment).value_counts(normalize=True) * 100,
    "% stopped buying (wk-59 segment)": fwd["% with no purchase in last 8 weeks"],
}).reindex(ORDER)
seg_tbl

,ชื่อกลุ่ม,households,% sales,median visits / wk,median CLV 12m $,% of campaign sends,% stopped buying (wk-59 segment)
Champions,ประจำ / มูลค่าสูง,916,70.54,1.80,"3,050.48",74.42,3.52
Loyal,ซื้อซ้ำ / มูลค่าปานกลาง,647,12.78,0.62,908.16,10.18,7.97
At Risk,เริ่มห่างหาย,575,15.05,0.63,740.02,15.22,17.34
Occasional,ซื้อเป็นครั้งคราว,362,1.63,0.17,110.13,0.18,35.00


In [4]:
hh = seg.monetary.sort_values(ascending=False)
act = lift_all[(lift_all["stratify on"] == "prepre") & (lift_all.window == "actual")].iloc[0]
plc = lift_all[(lift_all["stratify on"] == "prepre") & (lift_all.window == "placebo")].iloc[0]
bias = lift_all[(lift_all["stratify on"] == "pre")].set_index("window").lift_per_week

headline = pd.Series({
    "Total sales (2 yrs)": f"${tx.SALES_VALUE.sum() / 1e6:.2f}M",
    "Avg weekly sales (wk 17-102)": f"${weekly.loc[17:].mean() / 1e3:.0f}K  (H2 vs H1: {weekly.loc[60:].mean() / weekly.loc[17:59].mean() - 1:+.1%})",
    "Top 20% households share of sales": f"{hh.iloc[:500].sum() / hh.sum():.0%}",
    "Champions: % households / % sales": f"{seg_tbl.loc['Champions', 'households'] / 25:.0f}% / {seg_tbl.loc['Champions', '% sales']:.0f}%",
    "Week-59 Champions still Champions at end": f"{migr.loc['Champions', 'Champions']:.0f}%  (to At Risk: {migr.loc['Champions', 'At Risk']:.0f}%)",
    "Stopped buying: At Risk / Loyal / Champions": " / ".join(f"{fwd.loc[k, '% with no purchase in last 8 weeks']:.0f}%" for k in ["At Risk", "Loyal", "Champions"]),
    "Campaign sends going to Champions": f"{seg_tbl.loc['Champions', '% of campaign sends']:.0f}%",
    "Reached households never redeeming": f"{1 - redempt.household_key.nunique() / ct.household_key.nunique():.0%}",
    "Campaign lift (valid method)": f"{act.lift_per_week:+.2f} $/hh/wk  (95% CI {act.ci_low:+.2f} to {act.ci_high:+.2f}) · placebo {plc.lift_per_week:+.2f}",
    "Campaign lift (biased method, for contrast)": f"{bias['actual']:+.2f} actual vs {bias['placebo']:+.2f} placebo",
    "Sales on retail discount (excl. fuel)": f"{merch[merch.on_promo].SALES_VALUE.sum() / merch.SALES_VALUE.sum():.0%}",
    "Retail discount (excl. fuel) vs coupon discount": f"${merch.retail_disc.sum() / 1e6:.2f}M vs ${tx.coupon_disc.sum() / 1e3:.0f}K",
})
display(headline.to_frame("value"), lift_type.round(2))

,value
Total sales (2 yrs),$8.06M
Avg weekly sales (wk 17-102),$87K (H2 vs H1: +5.8%)
Top 20% households share of sales,53%
Champions: % households / % sales,37% / 71%
Week-59 Champions still Champions at end,85% (to At Risk: 13%)
Stopped buying: At Risk / Loyal / Champions,17% / 8% / 4%
Campaign sends going to Champions,74%
Reached households never redeeming,73%
Campaign lift (valid method),+0.76 $/hh/wk (95% CI -0.36 to +1.94) · place...
"Campaign lift (biased method, for contrast)",+6.18 actual vs +6.51 placebo


,actual_lift_per_week,actual_ci_low,actual_ci_high,placebo_lift_per_week,placebo_ci_low,placebo_ci_high,n_targeted_households
type,,,,,,,
TypeA,0.93,-0.71,2.69,0.26,-1.75,2.47,1513
TypeB,0.04,-1.25,1.58,0.56,-0.81,1.84,1023
TypeC,2.87,0.19,5.85,-2.16,-5.24,0.73,397


## Key Insights

**1. รายได้กระจุกตัวและโตช้า**
ยอดขาย $8.06M ใน 2 ปี ~$87K/สัปดาห์ โตเพียง ~6% ระหว่างครึ่งแรกกับครึ่งหลัง · ครัวเรือน top 20% ทำยอด ~53%
→ การเติบโตต้องมาจากการ **รักษาและเพิ่มมูลค่าลูกค้าเดิม** ไม่ใช่ลดราคาแข่ง

**2. Champions (37% ของครัวเรือน) ทำยอด ~71% และเสถียรที่สุด**
ซื้อเกือบ 2 ครั้ง/สัปดาห์ · CLV คาดการณ์ ~$3,050/ครัวเรือน/ปี · ทดสอบกับข้อมูลที่ไม่เคยเห็น พบว่า ~85% ยังเป็น Champions
แต่ **~13% หลุดไปเป็น At Risk** → รักษา และเฝ้าดูคนที่เริ่มห่าง · ส่วนหนึ่งของความภักดีมาจากการเติมน้ำมันที่ร้าน

**3. At Risk (23%) เป็นสัญญาณเตือนที่จริงแต่ไม่แม่นพอ**
ไม่ซื้อเลยใน 8 สัปดาห์สุดท้ายถึง ~17% (Loyal ~8%, Champions ~4%) แต่ **~83% ยังกลับมาเอง**
→ ส่งโปรแรงให้ทุกคนในกลุ่มจะส่งโดนคนที่จะกลับมาอยู่แล้ว · ใช้ **โมเดล churn คัดคนที่เสี่ยงจริง** (ขั้นต่อไป)
*หมายเหตุ: ยอดที่ลดลงของ At Risk ในครึ่งหลังเป็นผลของนิยามกลุ่ม จึงไม่ใช้เป็นหลักฐาน*

**4. แคมเปญกระจุกที่ลูกค้ายอดสูง**
74% ของการส่งไปที่ Champions, At Risk ~15%, Occasional แทบไม่ได้รับ · ครัวเรือนที่ได้รับ 73% ไม่เคยใช้คูปอง

**5. ข้อมูลยังยืนยันไม่ได้ว่าแคมเปญเพิ่มยอดซื้อ**
lift ที่ผ่าน placebo test = +$0.76/ครัวเรือน/สัปดาห์ (95% CI −0.4 ถึง +1.9) · บอกไม่ได้ว่าประเภทไหนดีกว่า
วิธีเทียบแบบไม่ผ่าน placebo ให้ผลบวกเกินจริง (+$6) → **ต้องใช้ holdout group แบบสุ่ม** อัตราใช้คูปอง (Type A 16%) ไม่ใช่ตัวชี้วัดผล

**6. ส่วนลดหน้าร้านเป็นต้นทุนก้อนใหญ่และไม่แยกกลุ่มลูกค้า**
~47% ของยอดขาย (ไม่นับน้ำมัน) ขายตอนลดราคา รวม ~$1.37M เทียบกับคูปอง ~$32K · ทุกกลุ่มอ่อนไหวเท่า ๆ กัน
→ **สมมติฐานที่ต้องทดสอบ:** เปลี่ยนส่วนลดหว่านบางส่วนเป็นข้อเสนอเฉพาะกลุ่ม (ข้อมูลนี้ยังไม่บอกว่าส่วนลดเพิ่มยอดหรือไม่)

**7. โครงสร้างหมวดของแต่ละกลุ่มนิ่ง**
GROCERY ~53–56% ในทุกกลุ่ม และเปลี่ยนไม่เกิน ~1 pp → ใช้ GROCERY เป็นหมวดหลักของโปรและการขายคู่

## ข้อเสนอทางการตลาดต่อแต่ละกลุ่ม

| กลุ่ม | เป้าหมาย | ข้อเสนอ | KPI |
|---|---|---|---|
| **Champions** — ประจำ/มูลค่าสูง | รักษา + เพิ่มมูลค่าตะกร้า | สิทธิพิเศษสมาชิก · โปรขายคู่ใน GROCERY ช่วง 15–19 น. · เฝ้าดูคนที่ห่างผิดปกติ | basket value, % ตกไป At Risk |
| **Loyal** — ซื้อซ้ำ | เพิ่มความถี่ | ทดสอบคูปอง "ซื้อครบ X ครั้งใน 4 สัปดาห์" ในหมวดที่ซื้อประจำ | visits/week |
| **At Risk** — เริ่มห่างหาย | ดึงกลับ *เฉพาะคนที่เสี่ยงจริง* | ใช้โมเดล churn คัดคน → ทดสอบข้อเสนอ GROCERY/MEAT เทียบกับ holdout | reactivation rate เทียบ holdout |
| **Occasional** — ครั้งคราว | ต้นทุนต่ำ | อีเมล/แอปเท่านั้น | cost per reactivation |

### วิธีวัดผลที่เสนอ (ทุกแคมเปญ)
1. สุ่ม **holdout 10–20%** ของกลุ่มเป้าหมายที่ไม่ได้รับแคมเปญ
2. วัด **ยอดซื้อส่วนเพิ่มต่อสัปดาห์** ของกลุ่มที่ได้รับเทียบ holdout ตลอด 4–8 สัปดาห์ ไม่ใช้อัตราใช้คูปองเป็นตัวชี้วัดหลัก
3. ไม่ส่งแคมเปญหลายตัวซ้อนกันให้คนกลุ่มเดียว เพื่อให้ผลแยกกันได้

### ขั้นต่อไป
- Market Basket Analysis — หาคู่สินค้าสำหรับโปรขายคู่
- Churn Prediction — ทำนายรายครัวเรือนว่าใครจะหยุดซื้อ (แยกช่วงสร้าง feature กับช่วงวัดผล)
- Dashboard สำหรับติดตามกลุ่มลูกค้าและผลแคมเปญ

### ข้อจำกัด
ไม่มีข้อมูลต้นทุน/กำไร (CLV เป็นเชิงรายได้แบบง่าย) · ข้อมูลประชากรมี 32% และเอนเอียงไปทาง Champions ·
ผลแคมเปญประมาณจากข้อมูลย้อนหลัง ไม่ใช่การทดลองแบบสุ่ม และแคมเปญทับซ้อนกัน